In [ ]:
import numpy as np
import sysl.symbolic as sls
import geolipi.symbolic as gls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html, create_multibuffer_shader_html
from IPython.display import display, HTML

settings = {
    "render_mode": "v4",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 2,
        "_RAYCAST_MAX_STEPS": 200,
    },
    "set_to_ubo": False,
    "export_params": False,
}



In [ ]:
# All 2D primitives expressions
all_primitives_2d_expressions = [
    gls.Circle2D((0.5,)),
    gls.RoundedBox2D((0.6, 0.8,), (0.05, 0.30, 0.5, 0.25,)),
    gls.Box2D((0.3, 0.1,)),
    gls.Rectangle2D((0.3, 0.1,)),
    gls.OrientedBox2D((0.0, 0.0,), (0.3, 0.1,), (0.05,)),
    gls.Rhombus2D((0.3, 0.1,)),
    gls.Trapezoid2D((0.3,), (0.1,), (0.2,)),
    gls.Parallelogram2D((0.3,), (0.2,), (0.1,)),
    gls.EquilateralTriangle2D((0.5,)),
    gls.IsoscelesTriangle2D((0.3, 0.2,)),
    gls.Triangle2D((0.0, 0.0,), (0.3, 0.0,), (0.15, 0.3,)),
    gls.UnevenCapsule2D((0.3,), (0.2,), (0.4,)),
    gls.RegularPentagon2D((0.5,)),
    gls.RegularHexagon2D((0.5,)),
    gls.RegularOctagon2D((0.5,)),
    gls.Hexagram2D((0.5,)),
    gls.Pentagram2D((0.5,),),
    gls.RegularStar2D((0.8,), (12,), (2,)),
    gls.Pie2D((0.5,), (0.8,)),
    gls.CutDisk2D((0.5,), (0.2,)),
    gls.Arc2D((0.5,), (0.3,), (0.1,)),
    gls.HorseShoe2D((0.5,), (0.3,), (0.1, 0.05,)),
    gls.Vesica2D((0.5,), (0.3,)),
    gls.OrientedVesica2D((0.0, 0.0,), (0.3, 0.1,), (0.1,)),
    gls.Moon2D((0.3,), (0.5,), (0.4,)),
    gls.RoundedCross2D((0.3,)),
    gls.Egg2D((0.3,), (0.1,), (0.2,), (0.4,)),
    gls.Heart2D(),
    gls.Cross2D((0.3, 0.1,), (0.05,)),
    gls.RoundedX2D((0.3,), (0.05,)),
    gls.Ellipse2D((0.3, 0.1,)),
    gls.BlobbyCross2D((0.3,)),
    gls.Tunnel2D((0.3, 0.1,)),
    gls.Stairs2D((0.3, 0.1,), 5),
    gls.QuadraticCircle2D(),
    gls.CoolS2D(),
    gls.CircleWave2D((0.5,), (0.3,)),
    gls.Segment2D((0.0, 0.0,), (0.3, 0.1,)),
    # These dont make sense for extrusion.
    # gls.Parabola2D((0.5,)),
    # gls.ParabolaSegment2D((0.3,), (0.1,)),
    # gls.Hyperbola2D((0.5,), (0.3,)),
    # gls.QuadraticBezierCurve2D((0.0, 0.0,), (0.3, 0.3,), (0.6, 0.0,)),
    # These don't have default_spec, so skipping for now:
    # Supported in Migumi currently. General Support TBD.
    # gls.PolyArc2D(((0.0, 0.0, 0.0,), (0.3, 0.1, 0.0,), (0.6, 0.0, 0.0,))),
    # gls.Polygon2D(((0.0, 0.0,), (0.3, 0.0,), (0.15, 0.3,))),
    
]


In [ ]:
SEL_INDEX = 15

# TO BE FIXED 16 26 30
# Remove 32
cur_expr = all_primitives_2d_expressions[SEL_INDEX]
# cur_expr = gls.Dilate3D(cur_expr, (0.05,))
# cur_expr = gls.SimpleExtrusion3D(cur_expr, (0.5,))
cur_expr = gls.SimpleRevolution3D(cur_expr, (0.1,))
# cur_expr = gls.Dilate3D(cur_expr, (0.05,))
print(cur_expr)
material = sls.MatRefV4("MatWood")
scene_with_material = sls.MatSolid(cur_expr, material)

# Get Shader Code
shader_info = evaluate_to_shader(scene_with_material, settings=settings, 
                                 mode="multipass", post_process_shader=["all_outline_nobg"])
# TO visualize in a browser:
html_code = create_multibuffer_shader_html(shader_info, show_controls=True)
# To visualize inline in jupyter notebook:
with open("/users/aganesh8/data/aganesh8/projects/project_neo/old/new_renders/html/test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))
# display(HTML(jupy_wrapper_html))

In [ ]:
# Also check match between eval in render. 
from sysl.shader.utils.texture import recursive_encode_texture_tensor
from geolipi.torch_compute import Sketcher, recursive_evaluate
sketcher = Sketcher(n_dims=3, resolution=64)
secondary_sketcher = Sketcher(n_dims=2, resolution=128)

# cur_expr = gls.ArbitraryCappedCylinder3D((0.3, 0.0, 0.1, ), (0.8, 0.0, 0.7,), (0.1,))
eval_out = recursive_evaluate(cur_expr.tensor(), sketcher, secondary_sketcher)# [..., 0]
if len(eval_out.shape) == 2:
    eval_out = eval_out[..., 0]
expr = gls.SDFGrid3D(eval_out, "torch_eval", (0.2,))

expr = recursive_encode_texture_tensor(expr, sketcher)

new_expr = sls.MatSolidV1(expr, material)

final_scene = gls.Union(
    new_expr,
    gls.Translate3D(scene_with_material, (1.0, 0.0, 0.0))
)


shader_code, uniforms, textures = evaluate_to_shader(final_scene, settings=settings)
# TO visualize in a browser:
with open("shader_code.glsl", "w") as f:
    f.write(shader_code)
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=False)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))
